<a href="https://colab.research.google.com/github/marviiiin/Reproducibility-Learning_To_Edit-/blob/main/LTE_Final_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LTE-LoRA: Training + Evaluation (Final)

**Runtime:** A100 GPU (Runtime > Change runtime type > A100)

Run every cell top-to-bottom. Training takes ~4-8 hours.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi | head -12
!python --version

Sat Apr 25 17:47:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Cell 2: Install dependencies
!pip install peft bitsandbytes trl datasets sentence_transformers higher iopath -q
print('Done!')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.8 MB/s eta 0:00:00
Done!


In [ ]:
# Cell 3: Clone repo + download data
import os
os.chdir('/content')
!rm -rf LTE
!git clone https://github.com/YJiangcm/LTE.git
os.chdir('/content/LTE')

from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='YuxinJiang/LTE_train_data', filename='LTE_train_data.json',
                repo_type='dataset', local_dir='data/')

# Subsample to 5K
import json
with open('data/LTE_train_data.json') as f:
    all_data = json.load(f)
subset = all_data[:5000]
with open('data/LTE_train_data_5k.json', 'w') as f:
    json.dump(subset, f)
print(f'Saved {len(subset)} training examples')


Cloning into 'LTE'...
remote: Enumerating objects: 939, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 939 (delta 55), reused 31 (delta 31), pack-reused 877 (from 1)
Receiving objects: 100% (939/939), 35.24 MiB | 28.78 MiB/s, done.
Resolving deltas: 100% (228/228), done.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


LTE_train_data.json:   0%|          | 0.00/105M [00:00<?, ?B/s]

Saved 5000 training examples


In [ ]:
# Cell 4: Pre-download the model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL = 'NousResearch/Llama-2-7b-chat-hf'
tok = AutoTokenizer.from_pretrained(MODEL)
m = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)
print(f'Model: {m.num_parameters()/1e9:.1f}B params')
del m, tok
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model: 6.7B params


In [12]:
# Cell 5: TRAIN with SFTTrainer (bypasses broken FastChat preprocessing)
import os, json, torch
os.chdir('/content/LTE')
os.environ['WANDB_DISABLED'] = 'true'

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained('NousResearch/Llama-2-7b-chat-hf')
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    'NousResearch/Llama-2-7b-chat-hf',
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='sdpa'
)

# Load and format data as LLaMA-2-Chat conversations
with open('data/LTE_train_data_5k.json') as f:
    raw_data = json.load(f)

def format_example(ex):
    text = ""
    for turn in ex['conversations']:
        if turn['from'] == 'human':
            text += f"[INST] {turn['value']} [/INST] "
        elif turn['from'] == 'gpt':
            text += f"{turn['value']} </s>"
    return {"text": text.strip()}

dataset = Dataset.from_list([format_example(ex) for ex in raw_data])
print(f"Dataset: {len(dataset)} examples")
print(f"Sample (first 300 chars): {dataset[0]['text'][:300]}")

# LoRA config (same as paper: r=8, alpha=16, dropout=0.05)
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

# Training config
training_args = SFTConfig(
    output_dir='output_lte_lora_llama-2_7b_chat',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=3e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=1,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to='none',
    dataset_text_field='text',
)

# Train
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model('output_lte_lora_llama-2_7b_chat')
print('Training complete! Model saved.')

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Dataset: 5000 examples
Sample (first 300 chars): [INST] Please acknowledge the updated information provided below and respond to the subsequent query.

[Updated Information]:
1. What is the native language of Christiane Cohendy? German
2. The nationality of Christiane Martel is? German
3. What is the native language of Alain Cuny? German

[Query]:


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,4.300152
2,4.292284
3,4.351595
4,4.368273
5,3.885861
6,3.701421
7,3.412248
8,3.084834
9,2.732777
10,2.812516


Training complete! Model saved.


In [13]:
# Cell 6: Verify checkpoint exists
import os
out = '/content/LTE/output_lte_lora_llama-2_7b_chat'
if os.path.exists(os.path.join(out, 'adapter_config.json')):
    print('Training complete! LoRA adapter saved.')
    for f in os.listdir(out):
        full = os.path.join(out, f)
        if os.path.isfile(full):
            print(f'  {f} ({os.path.getsize(full)/1e6:.1f} MB)')
else:
    print('WARNING: No adapter_config.json found!')
    print('Files:', os.listdir(out) if os.path.exists(out) else 'dir missing')

Training complete! LoRA adapter saved.
  training_args.bin (0.0 MB)
  README.md (0.0 MB)
  tokenizer.json (3.6 MB)
  tokenizer_config.json (0.0 MB)
  adapter_model.safetensors (33.6 MB)
  adapter_config.json (0.0 MB)


In [14]:
# Cell 7: Save checkpoint to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/LTE_checkpoints
!cp -r /content/LTE/output_lte_lora_llama-2_7b_chat /content/drive/MyDrive/LTE_checkpoints/
print('Saved to Drive!')

Mounted at /content/drive
Saved to Drive!


---
## Evaluation (Day 3)

In [15]:
# Cell 8: Download retrieval models
import os
os.chdir('/content/LTE')
!git clone https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 hugging_cache/all-MiniLM-L6-v2 2>/dev/null
print('Retrieval model ready.')

Retrieval model ready.


In [16]:
# Cell 9: Update hparams for evaluation
import yaml

hp = '/content/LTE/EasyEdit/hparams/IKE/llama-7b.yaml'
with open(hp) as f:
    cfg = yaml.safe_load(f)

cfg['model_name'] = 'NousResearch/Llama-2-7b-chat-hf'
cfg['lora_name'] = '/content/LTE/output_lte_lora_llama-2_7b_chat'
cfg['sentence_model_name'] = '/content/LTE/hugging_cache/all-MiniLM-L6-v2'

with open(hp, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Hparams updated:', cfg)

Hparams updated: {'alg_name': 'IKE', 'model_name': 'NousResearch/Llama-2-7b-chat-hf', 'sentence_model_name': '/content/LTE/hugging_cache/all-MiniLM-L6-v2', 'lora_name': '/content/LTE/output_lte_lora_llama-2_7b_chat', 'device': 0, 'results_dir': './results', 'k': 16, 'model_parallel': False}


In [17]:
# Cell 10: Evaluate on ZsRE
import os
os.chdir('/content/LTE/EasyEdit')
os.makedirs('output', exist_ok=True)

for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs']:
    print(f'\n{"="*50} ZsRE - {aspect} {"="*50}')
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/ZsRE/ZsRE-test-all.json \
        --eval_aspect=$aspect \
        --fluency


================================================== ZsRE - portability_r ==================================================
/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
Traceback (most recent call last):
  File "/content/LTE/EasyEdit/run_knowedit.py", line 6, in <module>
    from easyeditor import (
  File "/content/LTE/EasyEdit/easyeditor/__init__.py", line 1, in <module>
    from .dataset import *
  File "/content/LTE/EasyEdit/easyeditor/dataset/__init__.py", line 1, in <module>
    from .counterfact import CounterFactDataset
  File "/content/LTE/EasyEdit/easyeditor/dataset/counterfact.py", line 11, in <module>
    from ..trainer.utils import dict_to
  File "/content/LTE/EasyEdit/easyeditor/trainer/__init__.py", line 5, in <module>
    from .blip2_models import *
  File "/c

In [11]:
# Cell 11: Evaluate on WikiDatacounterfact
import os
os.chdir('/content/LTE/EasyEdit')

for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs', 'locality_f']:
    print(f'\n{"="*50} WikiData_cf - {aspect} {"="*50}')
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/wiki_counterfact/test_cf.json \
        --eval_aspect=$aspect \
        --fluency


================================================== WikiData_cf - portability_r ==================================================
/content/LTE/EasyEdit/easyeditor/trainer/algs/higher_utils/utils.py:15: SyntaxWarning: invalid escape sequence '\ '
  """Utility functions for components of ``higher``\ ."""
/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
Traceback (most recent call last):
  File "/content/LTE/EasyEdit/run_knowedit.py", line 6, in <module>
    from easyeditor import (
  File "/content/LTE/EasyEdit/easyeditor/__init__.py", line 1, in <module>
    from .dataset import *
  File "/content/LTE/EasyEdit/easyeditor/dataset/__init__.py", line 1, in <module>
    from .counterfact import CounterFactDataset
  File "/content/LTE/EasyEdit/easyeditor/dataset/counterfact.py", line 

In [18]:
# Cell 12: Collect & display all results
import json, os

output_dir = '/content/LTE/EasyEdit/output'
print('\n' + '='*60)
print('LTE-LoRA RESULTS')
print('='*60)

for fname in sorted(os.listdir(output_dir)):
    if fname.endswith('.json'):
        with open(os.path.join(output_dir, fname)) as f:
            metrics = json.load(f)
        print(f'\n{fname}: {len(metrics)} examples')

        # Edit success
        try:
            es = sum(m['post']['rewrite_acc'] for m in metrics) / len(metrics) * 100
            print(f'  Edit Success: {es:.2f}%')
        except: pass

        # Portability
        try:
            port = metrics[0]['post'].get('portability', {})
            if port:
                for k in port:
                    avg = sum(m['post']['portability'][k] for m in metrics) / len(metrics) * 100
                    print(f'  Portability ({k}): {avg:.2f}%')
        except: pass

        # Locality
        try:
            loc = metrics[0]['post'].get('locality', {})
            if loc:
                for k in loc:
                    avg = sum(m['post']['locality'][k] for m in metrics) / len(metrics) * 100
                    print(f'  Locality ({k}): {avg:.2f}%')
        except: pass

        # Fluency
        try:
            if 'fluency' in metrics[0]['post']:
                fl = sum(m['post']['fluency']['ngram_entropy'] for m in metrics) / len(metrics) * 100
                print(f'  Fluency: {fl:.2f}')
        except: pass

# Save to Drive
!cp -r /content/LTE/EasyEdit/output/* /content/drive/MyDrive/LTE_checkpoints/results/ 2>/dev/null
print('\nResults saved to Drive!')


LTE-LoRA RESULTS

Results saved to Drive!
